In [1]:
import os
import time
import random

import numpy as np
import pandas as pd

import torch
import torch.nn as nn
import torch.nn.functional as F

from torch.utils.data import Dataset, DataLoader

from skimage.metrics import (
    peak_signal_noise_ratio,
    structural_similarity,
    mean_squared_error
)


print(torch.__version__)


device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("Device:", device)

cuda


In [17]:
PROJECT_ROOT = "/home/jupyter-svecwai/Subbarao/semicon/KLA-Semiconductor-Image-Restoration"


MODEL_ROOT = os.path.join(
    PROJECT_ROOT,
    "models"
)


DATA_ROOT = os.path.join(
    PROJECT_ROOT,
    "data"
)


print(MODEL_ROOT)

/home/jupyter-svecwai/Subbarao/semicon/KLA-Semiconductor-Image-Restoration/models


In [19]:
GT_PATH = "../data/train/GT"
NOISY_PATH = "../data/train/NoisyLR"

files = sorted([
    f for f in os.listdir(GT_PATH)
    if f.endswith(".npy")
])

print("Total pairs:", len(files))

Total pairs: 3200


In [20]:
np.random.seed(SEED)

indices = np.random.permutation(
    len(files)
)

train_end = int(
    0.80 * len(files)
)

val_end = int(
    0.90 * len(files)
)

train_files = [
    files[i]
    for i in indices[:train_end]
]

val_files = [
    files[i]
    for i in indices[
        train_end:val_end
    ]
]

test_files = [
    files[i]
    for i in indices[
        val_end:
    ]
]

print("Train:", len(train_files))
print("Validation:", len(val_files))
print("Test:", len(test_files))

NameError: name 'SEED' is not defined

In [18]:
class KLADataset(Dataset):

    def __init__(
        self,
        lr_path,
        gt_path,
        files
    ):

        self.lr_path = lr_path
        self.gt_path = gt_path
        self.files = files


    def __len__(self):

        return len(self.files)


    def __getitem__(self,idx):

        name = self.files[idx]


        lr = np.load(
            os.path.join(
                self.lr_path,
                name
            )
        )

        gt = np.load(
            os.path.join(
                self.gt_path,
                name
            )
        )


        lr = torch.tensor(
            lr,
            dtype=torch.float32
        ).unsqueeze(0)


        gt = torch.tensor(
            gt,
            dtype=torch.float32
        ).unsqueeze(0)


        return lr,gt

NameError: name 'Dataset' is not defined

In [10]:
MODEL_ROOT = "/home/jupyter-svecwai/Subbarao/semicon/KLA-Semiconductor-Image-Restoration/models"


MODEL_PATHS = {

    "SRCNN":
    f"{MODEL_ROOT}/srcnn/srcnn_best.pth",

    "DnCNN":
    f"{MODEL_ROOT}/dncnn/dncnn_best.pth",

    "EDSR":
    f"{MODEL_ROOT}/edsr/edsr_best.pth",

    "NAF-SR":
    f"{MODEL_ROOT}/nafsr/nafsr_best.pth",

    "Residual U-Net":
    f"{MODEL_ROOT}/resunet_sr/resunet_sr_best.pth",

    "DA Residual U-Net":
    f"{MODEL_ROOT}/da_resunet_sr/da_resunet_sr_best.pth",

    "DA Residual U-Net v2":
    f"{MODEL_ROOT}/da_resunet_sr_v2/da_resunet_sr_v2_best.pth",

    "RLFN":
    f"{MODEL_ROOT}/rlfn/rlfn_best.pth",

    "DA-NAF Lite ResUNet":
    f"{MODEL_ROOT}/da_naf_lite_resunet_best.pth"
}


for name,path in MODEL_PATHS.items():

    print(
        f"{name:25} --> {os.path.exists(path)}"
    )

SRCNN                     --> True
DnCNN                     --> True
EDSR                      --> True
NAF-SR                    --> True
Residual U-Net            --> True
DA Residual U-Net         --> True
DA Residual U-Net v2      --> True
RLFN                      --> True
DA-NAF Lite ResUNet       --> True


In [13]:
import torch
import torch.nn as nn
import torch.nn.functional as F

import os
import time
import pandas as pd
import numpy as np


device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print(device)

cuda


In [14]:
# SRCNN class from 03_baseline_srcnn.ipynb

class SRCNN(nn.Module):

    def __init__(self):
        super().__init__()

        self.net = nn.Sequential(
            nn.Conv2d(1,64,9,padding=4),
            nn.ReLU(),

            nn.Conv2d(64,32,5,padding=2),
            nn.ReLU(),

            nn.Conv2d(32,1,5,padding=2)
        )


    def forward(self,x):

        return torch.clamp(
            self.net(x),
            0,
            1
        )

In [15]:
MODEL_PATHS = {

"SRCNN":
"/home/jupyter-svecwai/Subbarao/semicon/KLA-Semiconductor-Image-Restoration/models/srcnn/srcnn_best.pth",

"DnCNN":
"/home/jupyter-svecwai/Subbarao/semicon/KLA-Semiconductor-Image-Restoration/models/dncnn/dncnn_best.pth",

"EDSR":
"/home/jupyter-svecwai/Subbarao/semicon/KLA-Semiconductor-Image-Restoration/models/edsr/edsr_best.pth",

"NAF-SR":
"/home/jupyter-svecwai/Subbarao/semicon/KLA-Semiconductor-Image-Restoration/models/nafsr/nafsr_best.pth",

"Residual U-Net":
"/home/jupyter-svecwai/Subbarao/semicon/KLA-Semiconductor-Image-Restoration/models/resunet_sr/resunet_sr_best.pth",

"DA Residual U-Net":
"/home/jupyter-svecwai/Subbarao/semicon/KLA-Semiconductor-Image-Restoration/models/da_resunet_sr/da_resunet_sr_best.pth",

"DA Residual U-Net v2":
"/home/jupyter-svecwai/Subbarao/semicon/KLA-Semiconductor-Image-Restoration/models/da_resunet_sr_v2/da_resunet_sr_v2_best.pth",

"RLFN":
"/home/jupyter-svecwai/Subbarao/semicon/KLA-Semiconductor-Image-Restoration/models/rlfn/rlfn_best.pth",

"DA-NAF Lite ResUNet":
"/home/jupyter-svecwai/Subbarao/semicon/KLA-Semiconductor-Image-Restoration/models/da_naf_lite_resunet_best.pth"

}

In [16]:
loaded_models={}


for name,model in MODELS.items():

    model.load_state_dict(
        torch.load(
            MODEL_PATHS[name],
            map_location=device
        )
    )

    model.to(device)
    model.eval()

    loaded_models[name]=model

    print(name,"loaded")

NameError: name 'MODELS' is not defined